# Surface-code decoding demo
Mirrors `demo.py`: PyMatching vs a small MLP and CNN on Stim data. Run `python run_all.py --quick` first so checkpoints exist.

In [ ]:
import numpy as np
from src.circuits import make_circuit, circuit_stats
from src.data import sample
from src import mwpm
from src.train import load_trained, predict_logits
import make_report
d, p = 3, 0.01
circuit = make_circuit(d, p)
circuit_stats(circuit)

## One shot
The detection events are the only information the decoder gets.

In [ ]:
det, obs = sample(circuit, 64, seed=4000)
i = int(np.argmax(det.sum(1) > 0))
shot, truth = det[i], bool(obs[i])
print('fired detectors:', np.flatnonzero(shot), ' true flip:', truth)

## Decode it with all three decoders

In [ ]:
preds = {'PyMatching': bool(mwpm.decode(mwpm.build_matcher(circuit), shot[None])[0])}
for k in ('mlp', 'cnn'):
    preds[k.upper()] = bool(predict_logits(load_trained(k, d), shot[None])[0] > 0)
from src.viz import draw_shot
import config
from IPython.display import Image
Image(str(draw_shot(circuit, shot, truth, preds, p, d, config.FIGURES / 'demo_shot_nb.png')))

## Mini benchmark on fresh shots

In [ ]:
for bp in (0.003, 0.006, 0.010):
    c = make_circuit(d, bp)
    bd, bo = sample(c, 20_000, seed=4001)
    res = {'none': bo.mean(), 'PyMatching': (mwpm.decode(mwpm.build_matcher(c), bd) != bo).mean()}
    for k in ('mlp', 'cnn'):
        res[k] = ((predict_logits(load_trained(k, d), bd) > 0) != bo).mean()
    print(bp, {k: round(float(v), 5) for k, v in res.items()})

## Full results and conclusion (from `results/`)

In [ ]:
from IPython.display import display
display(Image('figures/fig1_ler_vs_p_d3.png'), Image('figures/fig6_throughput.png'))
print('\n'.join(make_report.conclusion_lines()))